<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## CMTC: Introducción y Matriz Generadora Q
### T2.2 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. De CMTD a CMTC: motivación
2. Definición de Cadena de Markov de Tiempo Continuo
3. Tasas de transición qᵢⱼ y tiempo de permanencia
4. Matriz Generadora Q: construcción e interpretación
5. Diagrama de tasas (rate diagram)
6. Ejemplo: máquina con fallos y reparaciones
7. Construcción de Q con Python

## Motivación: de discreto a continuo
Las Cadenas de Markov de **Tiempo Discreto** (CMTD) asumen transiciones en instantes fijos.

En muchos sistemas reales los cambios ocurren en instantes **aleatorios**:
- Una máquina falla después de un tiempo aleatorio
- Un cliente llega al azar durante el día
- Un servidor termina su atención en tiempo variable

<div class='defn'>
Una <strong>Cadena de Markov de Tiempo Continuo</strong> (CMTC) es un proceso estocástico {X(t), t ≥ 0} con espacio de estados discreto S que satisface la <em>propiedad de Markov</em>:

$$P(X(t+s)=j \mid X(s)=i, X(u)=x_u, u < s) = P(X(t+s)=j \mid X(s)=i)$$
</div>

## Tasas de transición
<div class='defn'>
La <strong>tasa de transición</strong> del estado i al estado j (i ≠ j) es:

$$q_{ij} = \lim_{h \to 0} \frac{P(X(t+h)=j \mid X(t)=i)}{h} \geq 0$$
</div>

**Tiempo de permanencia en el estado i:**
El tiempo que el proceso permanece en i antes de saltar sigue una distribución **Exponencial** con parámetro:

$$q_i = \sum_{j \neq i} q_{ij}$$

Esta es la consecuencia directa de la propiedad sin memoria (conexión con el Proceso de Poisson).

## Matriz Generadora Q
<div class='defn'>
La <strong>Matriz Generadora Q</strong> (o matriz de tasas) es una matriz n×n donde:

$$Q_{ij} = \begin{cases} q_{ij} & \text{si } i \neq j \\ -q_i = -\sum_{j \neq i} q_{ij} & \text{si } i = j \end{cases}$$
</div>

**Propiedades de Q:**
- Todos los elementos fuera de la diagonal son ≥ 0
- La suma de cada **fila** es cero: Q · **1** = **0**
- Los elementos de la diagonal son negativos (o cero)
- Q determina completamente la dinámica de la CMTC

## Ejemplo: máquina con fallos y reparaciones
**Estados:** 0 = operativa, 1 = en reparación
**Parámetros:** tasa de fallo α, tasa de reparación β

**Diagrama de tasas:**
```
        α
   0 -------> 1
   0 <------- 1
        β
```

**Probabilidades de transición en h pequeño:**
- P(0→1 en h) = αh + o(h)
- P(1→0 en h) = βh + o(h)
- P(0→0 en h) = 1 − αh + o(h)
- P(1→1 en h) = 1 − βh + o(h)

$$Q = \begin{pmatrix} -\alpha & \alpha \\ \beta & -\beta \end{pmatrix}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Parámetros ──────────────────────────────
alpha = 0.5   # tasa de fallo (fallas/hora)
beta  = 2.0   # tasa de reparación (reparaciones/hora)

# ── Construcción de Q ────────────────────────
Q = np.array([[-alpha, alpha],
              [beta,  -beta]])

print("Matriz Generadora Q:")
print(Q)
print(f"
Verificación filas suman 0: {Q.sum(axis=1)}")
print(f"
Interpretación:")
print(f"  Tiempo promedio operativa:    E[T₀] = 1/α = {1/alpha:.2f} h")
print(f"  Tiempo promedio en reparación: E[T₁] = 1/β = {1/beta:.2f} h")

## Ejemplo extendido: 3 estados

In [ ]:
# Sistema con 3 modos: normal (0), degradado (1), fallo (2)
# Tasas: 0->1: 0.3, 0->2: 0.1, 1->0: 1.0, 1->2: 0.4, 2->1: 0.8

estados = ['Normal', 'Degradado', 'Fallo']
# Construir Q entrada por entrada
Q3 = np.zeros((3, 3))
tasas = {(0,1): 0.3, (0,2): 0.1,
         (1,0): 1.0, (1,2): 0.4,
         (2,1): 0.8}
for (i,j), q in tasas.items():
    Q3[i,j] = q
np.fill_diagonal(Q3, -Q3.sum(axis=1))  # diagonal = -suma de la fila

print("Matriz Q (3 estados):")
header = f"{'':12s}" + "".join(f"{e:>12}" for e in estados)
print(header)
for i, row in enumerate(Q3):
    print(f"{estados[i]:12s}" + "".join(f"{v:>12.3f}" for v in row))
print(f"
Verificación: filas = {Q3.sum(axis=1)}")

# Visualización del diagrama de tasas
fig, ax = plt.subplots(figsize=(8,5))
pos = [(0.15, 0.5), (0.5, 0.85), (0.85, 0.5)]
colors = ['#1A7A9A', '#C8961E', '#C0392B']
for i, (x, y) in enumerate(pos):
    ax.add_patch(plt.Circle((x, y), 0.08, color=colors[i], alpha=0.85, zorder=3))
    ax.text(x, y, estados[i], ha='center', va='center', color='white', fontweight='bold', fontsize=10, zorder=4)
for (i,j), q in tasas.items():
    x0,y0 = pos[i]; x1,y1 = pos[j]
    dx, dy = x1-x0, y1-y0
    ax.annotate('', xy=(x1-0.09*dx/abs(dx+1e-9), y1-0.09*dy/abs(dy+1e-9)),
                xytext=(x0+0.09*dx/abs(dx+1e-9), y0+0.09*dy/abs(dy+1e-9)),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
    mx, my = (x0+x1)/2, (y0+y1)/2
    ax.text(mx, my+0.05, f'q={q}', ha='center', fontsize=9, color='#333')
ax.set_xlim(0,1); ax.set_ylim(0.2,1.1); ax.axis('off')
ax.set_title('Diagrama de tasas — sistema 3 estados', fontweight='bold')
plt.tight_layout(); plt.show()

## Conclusiones
- Una CMTC extiende las cadenas de Markov al tiempo continuo usando la propiedad sin memoria.
- Los tiempos de permanencia en cada estado son Exponenciales con tasa qᵢ.
- La Matriz Generadora Q codifica toda la dinámica: qᵢⱼ en fuera-diagonal, −qᵢ en diagonal.
- La suma de cada fila de Q es siempre cero.
- A partir de Q se obtiene tanto el análisis transitorio (ecuaciones de Kolmogorov) como el estacionario.

**Siguiente tema:** Análisis Transitorio y Estado Estable de CMTC.